# Stg 데이터 적재

이 파일은 MySQL `bigcontest_raw` 데이터베이스에 적재된 원본 데이터를
`bigcontest_stg` 데이터베이스로 변환 및 적재하기 위한 파일입니다.

Stg 단계에서는 분석을 위한 본격적인 전처리를 수행하지 않고,
Raw 데이터의 구조와 값을 최대한 유지하면서 기본적인 데이터 타입을 정리합니다.

## 주요 내용
- 프로젝트 환경 및 MySQL 접속정보 설정
- `02_stg_create_load.sql` 실행
- `bigcontest_stg` 데이터베이스 연결
- STG 테이블 생성 여부 확인
- RAW와 STG의 적재 행 수 비교
- 카드 데이터 타입 변환 결과 확인

## 결과
- `bigcontest_stg.flow_age` 생성 및 적재
- `bigcontest_stg.flow_time` 생성 및 적재
- `bigcontest_stg.flow_wkdy` 생성 및 적재
- `bigcontest_stg.card_topic1` 생성 및 적재
- RAW와 STG의 행 수 일치 여부 확인
- 카드 데이터의 날짜, 결제금액, 결제건수 타입 변환 확인

## 1. Imports & Function Definition

In [1]:
import os
import pandas as pd
import pymysql

from pathlib import Path
from dotenv import load_dotenv

from pymysql.constants import CLIENT
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
def find_project_root(start=None):
    """
    .env 파일을 기준으로 프로젝트의 최상위 경로를 탐색합니다.

    args:
        - start: 프로젝트 루트 탐색을 시작할 경로
                 None인 경우 현재 작업 경로에서 탐색을 시작합니다.

    return:
        - .env 파일이 존재하는 프로젝트 루트의 Path 객체
    """
    # 현재 작업 경로 또는 지정된 경로에서 탐색 시작
    current = Path(start or Path.cwd()).resolve()

    # 상위 디렉터리로 이동하면서 .env 파일 탐색
    while current != current.parent:
        if (current / ".env").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "프로젝트 루트의 .env 파일을 찾을 수 없습니다."
    )


def execute_sql_file(sql_file, host, port, user, password):
    """
    SQL 파일을 읽어 MySQL 서버에서 실행합니다.

    args:
        - sql_file: 실행할 SQL 파일의 Path 객체
        - host: MySQL 서버 주소
        - port: MySQL 서버 포트
        - user: MySQL 사용자명
        - password: MySQL 비밀번호

    return:
        - 반환값 없음
    """
    # SQL 파일 존재 여부 확인
    if not sql_file.exists():
        raise FileNotFoundError(
            f"SQL 파일을 찾을 수 없습니다: {sql_file}"
        )

    # SQL 스크립트 읽기
    sql_script = sql_file.read_text(
        encoding="utf-8"
    )

    # 특정 데이터베이스를 지정하지 않고 MySQL 서버에 연결
    connection = pymysql.connect(
        host=host,
        port=port,
        user=user,
        password=password,
        charset="utf8mb4",
        autocommit=True,
        client_flag=CLIENT.MULTI_STATEMENTS
    )

    try:
        with connection.cursor() as cursor:
            # 여러 SQL 문을 포함한 스크립트 전체 실행
            cursor.execute(sql_script)

            # 남아 있는 결과 집합 처리
            while cursor.nextset():
                pass

    finally:
        connection.close()


def create_mysql_engine(host, port, user, password, database):
    """
    지정한 MySQL 데이터베이스에 연결할 SQLAlchemy Engine을 생성합니다.

    args:
        - host: MySQL 서버 주소
        - port: MySQL 서버 포트
        - user: MySQL 사용자명
        - password: MySQL 비밀번호
        - database: 연결할 데이터베이스명

    return:
        - 생성된 SQLAlchemy Engine 객체
    """
    # MySQL 접속 URL 생성
    db_url = URL.create(
        drivername="mysql+pymysql",
        username=user,
        password=password,
        host=host,
        port=port,
        database=database,
        query={"charset": "utf8mb4"}
    )

    # 연결 상태 확인 기능을 포함한 Engine 생성
    return create_engine(
        db_url,
        pool_pre_ping=True
    )

## 2. 프로젝트 환경 설정 및 Stg SQL 실행

프로젝트 루트와 `.env` 파일의 MySQL 접속정보를 불러온 뒤,
`02_stg_create_load.sql` 파일을 실행합니다.

SQL 파일에서는 `bigcontest_stg` 데이터베이스와 Stg 테이블을 생성하고,
`bigcontest_raw`의 데이터를 Stg로 적재합니다.

In [3]:
# 프로젝트 루트 탐색
PROJECT_ROOT = find_project_root()

# .env 환경변수 로드
load_dotenv(PROJECT_ROOT / ".env")

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/lee-jongyoon/Documents/bigcontest


In [4]:
# MySQL 접속정보 설정
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [5]:
# Stg 데이터베이스 생성 및 적재 SQL 파일 경로
STG_SQL_FILE = (
    PROJECT_ROOT
    / "sql"
    / "stg_create_load.sql"
)

print("SQL FILE:", STG_SQL_FILE)

SQL FILE: /Users/lee-jongyoon/Documents/bigcontest/sql/stg_create_load.sql


In [6]:
# Stg 데이터베이스 생성 및 Raw → Stg 적재
execute_sql_file(
    sql_file=STG_SQL_FILE,
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD
)

print("STG 데이터베이스 생성 및 적재 완료")

STG 데이터베이스 생성 및 적재 완료


## 3. Stg 데이터베이스 연결 및 테이블 확인

생성된 `bigcontest_stg` 데이터베이스에 연결하고,
Stg 적재 대상 테이블이 정상적으로 생성되었는지 확인합니다.

In [7]:
# 생성된 Stg 데이터베이스 연결
stg_engine = create_mysql_engine(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database="bigcontest_stg"
)

In [8]:
# 현재 연결된 데이터베이스 확인
with stg_engine.connect() as conn:
    database = conn.execute(
        text("SELECT DATABASE();")
    ).scalar()

print("Connected Database:", database)

Connected Database: bigcontest_stg


In [9]:
# Stg 데이터베이스에 생성된 테이블 확인
pd.read_sql(
    """
    SHOW TABLES;
    """,
    stg_engine
)

,Tables_in_bigcontest_stg
0,card_topic1
1,flow_age
2,flow_time
3,flow_wkdy


## 4. Raw / Stg 적재 결과 검증

Raw와 Stg의 테이블별 전체 행 수를 비교하여 데이터가 누락 없이 적재되었는지 확인합니다.

현재 Stg 단계에서는 중복 제거 및 행 단위 전처리를 수행하지 않으므로,
각 테이블의 Raw 행 수와 Stg 행 수는 동일해야 합니다.

In [10]:
# Raw 데이터베이스 연결
raw_engine = create_mysql_engine(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database="bigcontest_raw"
)

In [11]:
# Raw / Stg 비교 대상 테이블
tables = [
    "flow_age",
    "flow_time",
    "flow_wkdy",
    "card_topic1"
]

In [12]:
# 테이블별 Raw / Stg 행 수 비교
validation_results = []

for table_name in tables:

    raw_count = pd.read_sql(
        f"""
        SELECT COUNT(*) AS cnt
        FROM {table_name};
        """,
        raw_engine
    ).loc[0, "cnt"]

    stg_count = pd.read_sql(
        f"""
        SELECT COUNT(*) AS cnt
        FROM {table_name};
        """,
        stg_engine
    ).loc[0, "cnt"]

    validation_results.append({
        "table_name": table_name,
        "raw_count": raw_count,
        "stg_count": stg_count,
        "result": "OK" if raw_count == stg_count else "CHECK"
    })


validation_df = pd.DataFrame(
    validation_results
)

validation_df

,table_name,raw_count,stg_count,result
0,flow_age,596252,596252,OK
1,flow_time,663350,663350,OK
2,flow_wkdy,842073,842073,OK
3,card_topic1,1044710,1044710,OK


## 5. Stg 데이터 타입 변환 결과 확인

Stg 단계에서 기본 데이터 타입이 의도한 형태로 변환되었는지 확인합니다.

특히 Raw에서 문자열로 저장한 카드 데이터의 기준일자, 결제금액 및 결제건수가
각각 `DATE`, `BIGINT`, `BIGINT` 타입으로 변환되었는지 확인합니다.

In [13]:
# 카드 Stg 테이블 컬럼 및 데이터 타입 확인
pd.read_sql(
    """
    DESCRIBE card_topic1;
    """,
    stg_engine
)

,Field,Type,Null,Key,Default,Extra
0,raw_id,bigint(20) unsigned,NO,PRI,None,
1,TA_YMD,date,YES,,None,
2,TIME_GB,varchar(20),YES,,None,
3,MCT_SGG_CD,varchar(100),YES,,None,
4,MCT_RY_CD,varchar(100),YES,,None,
5,SEX_CCD,varchar(50),YES,,None,
6,AGE_CCD,varchar(50),YES,,None,
7,TS_AT,bigint(20),YES,,None,
8,USE_CNT,bigint(20),YES,,None,
9,source_file,varchar(255),NO,,None,


In [14]:
# 타입 변환이 적용된 카드 데이터 샘플 확인
pd.read_sql(
    """
    SELECT  raw_id
            ,TA_YMD
            ,TIME_GB
            ,MCT_SGG_CD
            ,MCT_RY_CD
            ,SEX_CCD
            ,AGE_CCD
            ,TS_AT
            ,USE_CNT
            ,source_file
            ,source_row_num
            ,raw_loaded_at
            ,stg_loaded_at
      FROM  card_topic1
     LIMIT  5;
    """,
    stg_engine
)

,raw_id,TA_YMD,TIME_GB,MCT_SGG_CD,MCT_RY_CD,SEX_CCD,AGE_CCD,TS_AT,USE_CNT,source_file,source_row_num,raw_loaded_at,stg_loaded_at
0,1,2025-07-04,12_17,서울 강남구,가구,법인,법인,15477163,16,신한카드_빅콘테스트2026_데이터1.txt,1,2026-09-22 09:37:27,2026-09-22 09:40:35
1,2,2025-07-21,12_17,서울 강남구,가구,법인,법인,308451,38,신한카드_빅콘테스트2026_데이터1.txt,2,2026-09-22 09:37:27,2026-09-22 09:40:35
2,3,2025-07-03,12_17,서울 강남구,가구,여성,20 대,78970,5,신한카드_빅콘테스트2026_데이터1.txt,3,2026-09-22 09:37:27,2026-09-22 09:40:35
3,4,2025-07-03,18_23,서울 강남구,가구,남성,20 대,271220,5,신한카드_빅콘테스트2026_데이터1.txt,4,2026-09-22 09:37:27,2026-09-22 09:40:35
4,5,2025-07-06,18_23,서울 강남구,가구,여성,20 대,260872,5,신한카드_빅콘테스트2026_데이터1.txt,5,2026-09-22 09:37:27,2026-09-22 09:40:35
